# Diabetes Model Saving & Prediction Pipeline

This notebook prepares the selected Diabetes classification pipeline for
reuse in the Healytics application.

The complete pipeline includes:

- Missing-value handling
- Feature scaling
- Machine learning model

The preprocessing and model are saved together so that future predictions
use exactly the same transformations as the training process.

The saved model will later be loaded by the Healytics application.

## 1. Import Required Libraries

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

## 2. Recreate the Diabetes Prediction Pipeline

The selected Logistic Regression pipeline is recreated using the same
preprocessing configuration used during model development and evaluation.

The pipeline contains median imputation, standardization, and Logistic
Regression.

In [2]:
diabetes_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=1,
        penalty="l2",
        max_iter=2000,
        random_state=42
    ))
])

## 3. Prepare the Training Data

The raw Diabetes dataset is loaded and the known zero-coded missing
measurements are converted to `NaN`.

Only the training data is used to fit the final saved pipeline.

In [4]:
df = pd.read_csv("../data/raw/diabetes.csv")

invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[invalid_zero_columns] = df[invalid_zero_columns].replace(0, np.nan)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (768, 8)
Target: (768,)


## 4. Train the Final Diabetes Pipeline

The selected pipeline is fitted on the complete Diabetes dataset.

After model selection and final evaluation, using all available labeled data
for the final saved training artifact allows the deployed model to learn from
all available training examples.

The previously evaluated test results remain documented separately and are
not changed by this final fitting step.

In [5]:
diabetes_pipeline.fit(X, y)

print("Final Diabetes pipeline trained successfully.")

Final Diabetes pipeline trained successfully.


c:\Users\mvans\OneDrive\Desktop\Healytics\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 5. Create Model Storage Directory

Trained model artifacts are stored in the project's `models` directory.

In [6]:
os.makedirs("../models", exist_ok=True)

print("Models directory ready.")

Models directory ready.


## 6. Save the Diabetes Pipeline

The complete preprocessing and prediction pipeline is saved as a Joblib
artifact.

Saving the complete pipeline prevents the application from having to
reimplement the preprocessing steps manually.

In [7]:
model_path = "../models/diabetes_pipeline.joblib"

joblib.dump(
    diabetes_pipeline,
    model_path
)

print(f"Model saved to: {model_path}")

Model saved to: ../models/diabetes_pipeline.joblib


In [8]:
print("File exists:", os.path.exists(model_path))

File exists: True


In [9]:
print("File size:", round(os.path.getsize(model_path) / 1024, 2), "KB")

File size: 2.22 KB


## 7. Reload the Saved Pipeline

The saved model is loaded from disk to verify that the artifact can be
successfully restored independently of the original Python object.

In [10]:
loaded_diabetes_pipeline = joblib.load(model_path)

print("Saved pipeline loaded successfully.")
print(type(loaded_diabetes_pipeline))

Saved pipeline loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


## 8. Test Prediction

A sample input containing all eight Diabetes model features is passed through
the reloaded pipeline.

This verifies that the saved pipeline can perform both preprocessing and
prediction.

In [11]:
sample_input = pd.DataFrame([{
    "Pregnancies": 2,
    "Glucose": 120,
    "BloodPressure": 70,
    "SkinThickness": 25,
    "Insulin": 100,
    "BMI": 30.5,
    "DiabetesPedigreeFunction": 0.45,
    "Age": 30
}])

prediction = loaded_diabetes_pipeline.predict(sample_input)

probability = loaded_diabetes_pipeline.predict_proba(sample_input)[0, 1]

print("Prediction:", prediction[0])
print("Diabetes probability:", round(probability, 4))

Prediction: 0
Diabetes probability: 0.2076


In [12]:
expected_features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

print("Expected features:")
print(expected_features)

print("\nSample features:")
print(list(sample_input.columns))

print(
    "\nFeature order correct:",
    list(sample_input.columns) == expected_features
)

Expected features:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Sample features:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Feature order correct: True


In [13]:
import sys

sys.path.append("../")

from src.prediction.diabetes_predictor import predict_diabetes

In [14]:
sample_input = {
    "Pregnancies": 2,
    "Glucose": 120,
    "BloodPressure": 70,
    "SkinThickness": 25,
    "Insulin": 100,
    "BMI": 30.5,
    "DiabetesPedigreeFunction": 0.45,
    "Age": 30
}

result = predict_diabetes(sample_input)

print(result)

{'prediction': 0, 'probability': 0.20757290800496994}


In [15]:
invalid_input = {
    "Pregnancies": 2,
    "Glucose": 120
}

try:
    predict_diabetes(invalid_input)
except ValueError as e:
    print("Validation error:", e)

Validation error: Missing required features: ['BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
